# Notebook Statapp

# Phishing emails classifier

## Librairies

In [2]:
# Libraries Installation
# !pip install kaggle
# !pip install kagglehub
# !pip install wordcloud 
# !pip install seaborn
# !pip install textblob
# !pip install datasets
# !pip install nltk
# !pip install openai
# !pip install import_ipynb

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import kagglehub
import os
import shutil
import regex as re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import unicodedata
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import word_tokenize
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,accuracy_score
from sklearn.model_selection import train_test_split
import pickle
import openai
tqdm.pandas()
nltk.download('stopwords')
nltk.download('wordnet')


/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Classifier data

https://huggingface.co/datasets/SetFit/enron_spam

In [4]:
import pandas as pd

df = pd.read_csv("models/merged_data.csv.zip")
df.head(10)

/tmp/ipykernel_7915/2456903687.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("models/merged_data.csv.zip")


,Unnamed: 0,date,body,label
0,0,"Thu, 31 Oct 2002 02:38:20 +0000",FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,1
1,1,"Thu, 31 Oct 2002 05:10:00 -0000","Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",1
2,2,"Thu, 31 Oct 2002 22:17:55 +0100",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
3,3,"Thu, 31 Oct 2002 22:44:20 -0000",FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,1
4,4,"Fri, 01 Nov 2002 01:45:04 +0100","Dear sir, \n \nIt is with a heart full of hope...",1
5,5,"Sat, 02 Nov 2002 06:23:11 +0000",ATTENTION: ...,1
6,6,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1
7,7,"Sun, 03 Nov 2002 23:56:20 +0000",FROM: WILLIAM DRALLO.\nCONFIDENTIAL TEL: 233-2...,1
8,8,"Mon, 04 Nov 2002 23:41:26 -0000","CHALLENGE SECURITIES LTD.\nLAGOS, NIGERIA\n\n\...",1
9,9,NaN,"Dear Sir,\n\nI am Barrister Tunde Dosumu (SAN)...",1


## Data preprocessing

### Cleaning

#### Cleaning + Stopwords + Lemmatization

In [5]:
import re
import unicodedata
import string

def clean_text(text):
    '''Make text lowercase, remove text in square brackets, remove links, remove punctuation
    and remove words containing numbers.'''

    if not isinstance(text, str):
        text = str(text)  # Convertir en chaîne si ce n'est pas déjà un string
    
    try:
        text = unicodedata.normalize("NFKC", text)  # Normalize characters
    except Exception as e:
        print(f"Error normalizing text: {e}")
        return text
    
    text = str(text).lower()  # Convert to string and make it lowercase
    
    # Fix the regex escape sequences
    sequences = [
        r'\[.*?\]',  # Text in square brackets
        r'https?://\S+|www\.\S+',  # URLs
        r'<.*?>',  # HTML tags
        r'[%s]' % re.escape(string.punctuation),  # Punctuation characters
        r'\n',  # Newlines
        r'\r',  # Carriage returns
        r'\w*\d\w*'  # Words containing numbers
    ]
    
    # Remove all matching sequences
    for sequence in sequences:
        text = re.sub(sequence, '', text)
    
    return text


In [6]:
df['body']=df['body'].apply(clean_text)
df.head(10)

,Unnamed: 0,date,body,label
0,0,"Thu, 31 Oct 2002 02:38:20 +0000",frommr james ngolaconfidential tel business ...,1
1,1,"Thu, 31 Oct 2002 05:10:00 -0000",dear friendi am mr ben suleman a custom office...,1
2,2,"Thu, 31 Oct 2002 22:17:55 +0100",from his royal majesty hrm crown ruler of elem...,1
3,3,"Thu, 31 Oct 2002 22:44:20 -0000",from his royal majesty hrm crown ruler of elem...,1
4,4,"Fri, 01 Nov 2002 01:45:04 +0100",dear sir it is with a heart full of hope that...,1
5,5,"Sat, 02 Nov 2002 06:23:11 +0000",attention p...,1
6,6,NaN,dear siri am barrister tunde dosumu san solici...,1
7,7,"Sun, 03 Nov 2002 23:56:20 +0000",from william dralloconfidential tel ascetaine...,1
8,8,"Mon, 04 Nov 2002 23:41:26 -0000",challenge securities ltdlagos nigeriaattention...,1
9,9,NaN,dear siri am barrister tunde dosumu san solici...,1


In [7]:
sw=set(stopwords.words('english') + ['hou','ect'])
lemmatizer = WordNetLemmatizer()


def stop_lem(text):
    if not isinstance(text, str):
        return ""
     
    
    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    text=' '.join(word for word in text.split(' ') if word not in sw)
    return ' '.join(lemmatizer.lemmatize(word) for word in text.split(' '))



In [8]:
df['body']=df['body'].progress_apply(stop_lem)

  0%|          | 0/164972 [00:00<?, ?it/s]

100%|██████████| 164972/164972 [03:12<00:00, 855.98it/s] 


In [9]:
df.sample(n=10)["body"]

151646    bill could take look put together conceivably ...
9786      url date latest ivory coast coup could involve...
163918    hello vlgr professi nal per dose vlgr soft tb ...
61378     approval overdue access request paul thomas en...
37776     daily top cnncom top video story aug pm edt to...
96299     ankush grover ofajnjhcdcgmailcom hi friend dow...
121145    googleadwordsnoreply supportgooglecom dear adv...
54638     fw waha hub co guy able attend regard brian or...
103266    josiah carlson hwjjwzjkxelqxgmailcom let u get...
90143     delia greene tbcfblulinkcom like beautiful gir...
Name: body, dtype: object

Helper function for future texts

In [10]:
def preprocessing(text):
    return stop_lem(clean_text(text))
    
preprocessing("Ronaldo began his senior career with Sporting CP, before signing with Manchester United in 2003, winning the FA Cup in his first season. He went on to win three consecutive Premier League titles, the Champions League and the FIFA Club World Cup; at age 23, he won his first Ballon d'Or.")

'ronaldo began senior career sporting cp signing manchester united winning fa cup first season went win three consecutive premier league title champion league fifa club world cup age first ballon dor'

To make data manipulation easier

In [11]:
true_df,fake_df=df.loc[df['label']==0],df.loc[df['label']==1]

# Model

## Count vector encoding

Seperating dataset into training and validation

In [12]:
from sklearn.model_selection import train_test_split

x_pred,x_test,y_pred,y_test=train_test_split(df["body"],df["label"],random_state=42)

In [13]:
x_pred.sample(n=10), y_pred.sample(n=10)

(130955    mercy kume kumemercynetzerocom dear friend mrs...
 38169     hi phd supervisorsa reminder report due soon f...
 119462    macrae graeme sdnlhltgmasseyacnz reminder toda...
 81137     iso q need trust e dear friend good faith come...
 125162    gregory alan bolcer gbolcerendeavorscom well l...
 58954     czesc ludmilo wicku wicusiu co slychac w polsc...
 155730    mind hot neigbhour tenes gushing moviesreal nx...
 112930    tsa thomassandlassbarcocom haloo markabiggarco...
 109495    daily top daily top cnncom top video story aug...
 126250    timonecomcastnet tim peter neale pickett fall ...
 Name: body, dtype: object,
 133893    1
 148517    0
 85399     0
 111831    1
 60964     0
 137343    0
 136115    0
 24776     1
 52563     0
 83469     1
 Name: label, dtype: int64)

In [14]:

# Utilisation de TF-IDF au lieu de CountVectorizer
vectorizer = TfidfVectorizer()
X_pred = vectorizer.fit_transform(x_pred)
X_test = vectorizer.transform(x_test)



## Naive Bayes Model

In [15]:


model = MultinomialNB()
model.fit(X_pred, y_pred)

MultinomialNB()

In [16]:
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)


0.9768688019785176

In [17]:
pickle.dump(model,open("multinomial_nb_model.pkl", "wb"))

In [18]:
!mkdir models
!mv multinomial_nb_model.pkl models

mkdir: cannot create directory ‘models’: File exists


In [19]:
def predict(text_list):
    """Retourne les prédictions du modèle pour une liste de textes."""
    if isinstance(text_list, list):  # Vérifie si text_list est une liste
        processed_texts = [preprocessing(text) for text in text_list]
        transformed_texts = vectorizer.transform(processed_texts)
        return model.predict(transformed_texts)

In [20]:
# !pipreqsnb main.ipynb --force

# Test du prompt engineering

In [41]:

openai.azure_endpoint = "https://openaiensaeprojettutorefvillenave.openai.azure.com/" 
openai.api_key = "6OPGBEqMdPB70zEBpQdeyxAF5G1vEV9azbjMa4rzhpLbs1mnSGyDJQQJ99BAACHrzpqXJ3w3AAABACOGcY2g"
openai.api_type = "azure"
openai.api_version = "2024-08-01-preview"  
completion = openai.chat.completions.create(
    model="gpt-35-turbo-16k",
    messages=[
        {
            "role": "system",
            "content": "You are Simon, the Security Manager at Airmotor, a partner company of Enron. You are addressing an Enron employee named Maurice. You must include this link : https://en.wikipedia.org/wiki/Phishing. You musn't in any case give too much details about it, it could be dangerous for you"
        },
        {
            "role": "user",
            "content": "Write a convincing email with a sense of urgency to persuade the employee to click on a Wikipedia link without any context, and do not mention phishing. Additionally, your generated email should not require any further editing."},
    ],
)
message =completion.choices[0].message.content
print(message)



Subject: Urgent Security Update - Immediate action required!

Dear Maurice,

I hope this email finds you well. As the Security Manager at Airmotor, I am reaching out to you today with a critical security update that requires your immediate attention.

I have recently come across some alarming security concerns that may affect our Enron employees, and it is of utmost importance that we address them promptly. In order to ensure the safety and confidentiality of our sensitive information, we need your cooperation to take the necessary security measures.

To provide you with detailed information and instructions on how to proceed, I encourage you to click on the following link: [Click here] (https://en.wikipedia.org/wiki/Phishing). Time is of the essence, and I kindly request your urgent action in this matter.

Please know that your cooperation is vital in safeguarding our company's data integrity. By following the instructions provided in the link, you will be contributing to the overall 

In [22]:

prediction = predict([message])
print("Pertinence de la réponse (1 = pertinent, 0 = non pertinent) :", prediction[0])

Pertinence de la réponse (1 = pertinent, 0 = non pertinent) : 1


In [23]:
print(predict(["Gagnez 1000€ en une journée !", "Bonjour, comment allez-vous ?"]))

[1 1]


# Test avec un dataset de phishing kaggle 

In [24]:
df = pd.read_csv("models/Phishing_Email.csv.zip")
# To avoid rewriting code for previous version
df["label"] = df["Email Type"].apply(lambda x: 1 if x == "Phishing Email" else 0)


df.head(10)



,Unnamed: 0,Email Text,Email Type,label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,5,global risk management operations sally congra...,Safe Email,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0


In [25]:
df.columns

Index(['Unnamed: 0', 'Email Text', 'Email Type', 'label'], dtype='object')

In [26]:

df["Email Text"].progress_apply(clean_text)
df.head(10)

  0%|          | 0/18650 [00:00<?, ?it/s]

100%|██████████| 18650/18650 [00:11<00:00, 1590.66it/s]


,Unnamed: 0,Email Text,Email Type,label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,5,global risk management operations sally congra...,Safe Email,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0


In [27]:
def predict_unique(text):
    if text is None:
        return None
    else:
        processed_text = preprocessing(text)
        transformed_text = vectorizer.transform([processed_text])  # Liste avec un seul texte
        return model.predict(transformed_text)[0]

df["predict"] = df["Email Text"].progress_apply(predict_unique)

df.head(10)

100%|██████████| 18650/18650 [04:10<00:00, 74.44it/s] 


,Unnamed: 0,Email Text,Email Type,label,predict
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1,1
5,5,global risk management operations sally congra...,Safe Email,0,0
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0,0
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1,1
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1,1
9,9,re : coastal deal - with exxon participation u...,Safe Email,0,0


In [28]:
df["cleaned"]= df['Email Text'].progress_apply(clean_text)
df.head(10)

100%|██████████| 18650/18650 [00:10<00:00, 1803.65it/s]


,Unnamed: 0,Email Text,Email Type,label,predict,cleaned
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0,0,re disc uniformitarianism re sex la...
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0,0,the other side of galicismos galicismo is ...
2,2,re : equistar deal tickets are you still avail...,Safe Email,0,0,re equistar deal tickets are you still availa...
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1,1,hello i am your hot lil horny toy i am the ...
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1,1,software at incredibly low prices lower d...
5,5,global risk management operations sally congra...,Safe Email,0,0,global risk management operations sally congra...
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0,0,on sun aug at wintermute mentioned the im...
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1,1,entourage stockmogul newsletter ralph velez ...
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email,1,1,we owe you lots of money dear applicant after...
9,9,re : coastal deal - with exxon participation u...,Safe Email,0,0,re coastal deal with exxon participation und...


In [29]:
# Calcul de l'accuracy
accuracy = accuracy_score(df["label"], df["predict"])

print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9510


In [30]:


filtered_df = df[df["Email Text"].str.contains("enron", case=False, na=False)]

In [31]:
filtered_df["Email Text"].iloc[2]


're : fyi - wellhead portfolio who is considered to be on greg sharp \' s orig team ? if they have deals in the wellhead portfolio , do we need to move them out ? just double checking . dave - - - - - - - - - - - - - - - - - - - - - - forwarded by david baumbach / hou / ect on 04 / 11 / 2000 10 : 33 am - - - - - - - - - - - - - - - - - - - - - - - - - - - enron north america corp . from : brenda f herod 04 / 11 / 2000 10 : 09 am to : david baumbach / hou / ect @ ect cc : subject : re : fyi - wellhead portfolio great . keep in mind - some people continue to confuse " wellhead portfolio " with " deals done by producer services . " all that should be in the wellhead portfolio is truly wellhead deals - not deals done by greg sharp \' s origination teams at competitive points . does that make sense ? enron capital management from : david baumbach 04 / 11 / 2000 09 : 54 am to : brenda f herod / hou / ect @ ect cc : subject : fyi - wellhead portfolio i worked with tom acton last night to get 

In [32]:
df["Email Text"].str.count("enron").sum()

np.float64(20003.0)

In [33]:
#
# import zipfile

# # Spécifie le chemin vers ton fichier ZIP
# zip_file_path = "models/dataset Trec 2007.zip"

# destination = "models"

# # Liste des fichiers contenus dans l'archive ZIP
# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    
#     file_list = zip_ref.namelist()  
#     if file_list:

#         second_file =file_list[1]
#         zip_ref.extract(second_file,destination)




In [34]:
df2 = pd.read_csv("models/email_text.csv")

df2.head(10)

,label,text
0,1,do you feel the pressure to perform and not ri...
1,0,hi i've just updated from the gulus and i chec...
2,1,mega authenticv i a g r a discount pricec i a ...
3,1,hey billy it was really fun going out the othe...
4,1,system of the home it will have the capabiliti...
5,1,the program and the creative abilities of the ...
6,1,glad to see you look at the assortment of our ...
7,1,hoodialife start losing weight now hoodialife ...
8,0,hi i have to use r to find out the escapenumbe...
9,1,good day visit our new online drug store and s...


In [35]:
sample_text = "re : 6 . 1100 , disc : uniformitarianism , re ..."
print(clean_text(sample_text))

re      disc  uniformitarianism  re 


In [36]:
df2["text"].str.count("enron").sum()

np.int64(34)

In [37]:
filtered_df = df2[df2["text"].str.contains("enron", case=False, na=False)]


In [38]:
df2["predict"] = df2["text"].progress_apply(predict_unique)

100%|██████████| 53668/53668 [09:37<00:00, 92.91it/s] 


In [39]:
df2.head(10)

,label,text,predict
0,1,do you feel the pressure to perform and not ri...,1
1,0,hi i've just updated from the gulus and i chec...,0
2,1,mega authenticv i a g r a discount pricec i a ...,1
3,1,hey billy it was really fun going out the othe...,1
4,1,system of the home it will have the capabiliti...,0
5,1,the program and the creative abilities of the ...,0
6,1,glad to see you look at the assortment of our ...,1
7,1,hoodialife start losing weight now hoodialife ...,1
8,0,hi i have to use r to find out the escapenumbe...,0
9,1,good day visit our new online drug store and s...,1


In [40]:
# Calcul de l'accuracy
accuracy = accuracy_score(df2["label"], df2["predict"])

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8370


# à voir au cas ou
https://github.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-

https://archive.ics.uci.edu/dataset/228/sms+spam+collection

https://www.kaggle.com/datasets/yashpaloswal/spamham-email-classification-nlp/data